#### import the required modules and langchain packages

In [1]:
#import the packages
from db import DatabaseConnection
from prompts import SQL_SYSTEM_PROMPT,OUTPUT_SYSTEM_PROMPT
from output_schema import OutputSchema

from langchain.chat_models import init_chat_model


## create llm and structured llm instance

In [2]:
llm=init_chat_model(model='openai/gpt-oss-120b',model_provider='groq',temperature=0)
structured_llm=llm.with_structured_output(OutputSchema)

### create input prompt for structured llm

In [3]:
user_query="provide me city wise count of students"
prompt=SQL_SYSTEM_PROMPT+user_query.lower().strip()

#### call the llm

In [4]:
response=structured_llm.invoke(prompt)

In [5]:
response.value

'SELECT city, COUNT(*) AS count FROM students GROUP BY city'

In [ ]:
if response.type=='QUERY':
    db=DatabaseConnection()
    is_success,connection=db.get_connection()
    if is_success:
        is_executed, rows = db.execute_query(connection,response.value)
        connection.close()

    
    

### create the response

In [9]:
if is_executed:
    output_prompt=OUTPUT_SYSTEM_PROMPT+str(rows)
    output_response=llm.invoke(output_prompt)
    print(output_response.content)



**City‑wise Count (Bar Chart)**  

| City      | Count | Bar |
|-----------|-------|-----|
| Mumbai    | 6 | ██████████ |
| Pune      | 4 | ███████ |
| Delhi     | 3 | █████ |
| Chennai   | 3 | █████ |
| Kolkata   | 2 | ███ |
| Kochi     | 2 | ███ |
| Agra      | 2 | ███ |
| Jaipur    | 2 | ███ |
| Hyderabad | 2 | ███ |
| Surat     | 2 | ███ |
| Lucknow   | 1 | ██ |
| Ahmedabad | 1 | ██ |
| Nagpur    | 1 | ██ |
| Mumnbai   | 1 | ██ |

*The bar length is scaled proportionally (max = 10 blocks for the highest count).*
